# Use Logistic Regression to Predict Market Movement : Up/Down Based on Past Returns

In [3]:
### Reference https://github.com/Mashimo/datascience/blob/master/01-Regression/LogisticRegressionSM.ipynb

import pandas as pd
import csv


import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as seabornInstance 
import statsmodels.api as sm

import statsmodels.formula.api as smf


from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn import metrics

In [4]:
#read Default data
Smarket = pd.read_csv('H:\My Documents\TEACHING\Financial_Data_Analytics\MS_FINANCE_REVISED\DATA\Smarket.csv')

In [5]:
#Droppong column 0 as it is not needed and storing the data in a new object d2
m2=Smarket.drop(Smarket.columns[0], axis=1)
m2.head()

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,2001,0.381,-0.192,-2.624,-1.055,5.010,1.1913,0.959,Up
1,2001,0.959,0.381,-0.192,-2.624,-1.055,1.2965,1.032,Up
2,2001,1.032,0.959,0.381,-0.192,-2.624,1.4112,-0.623,Down
3,2001,-0.623,1.032,0.959,0.381,-0.192,1.2760,0.614,Up
4,2001,0.614,-0.623,1.032,0.959,0.381,1.2057,0.213,Up


In [6]:
### WE want to keep the categorical variable and create a dummy variable for that in a seperate column ####

Direction_dummy=pd.get_dummies(m2['Direction'])

Direction_dummy.columns =['Direction_Down', 'Direction_Up'] 
Direction_dummy.head()

m3=pd.concat([m2,Direction_dummy], axis=1)
m3.head()


,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction,Direction_Down,Direction_Up
0,2001,0.381,-0.192,-2.624,-1.055,5.010,1.1913,0.959,Up,0,1
1,2001,0.959,0.381,-0.192,-2.624,-1.055,1.2965,1.032,Up,0,1
2,2001,1.032,0.959,0.381,-0.192,-2.624,1.4112,-0.623,Down,1,0
3,2001,-0.623,1.032,0.959,0.381,-0.192,1.2760,0.614,Up,0,1
4,2001,0.614,-0.623,1.032,0.959,0.381,1.2057,0.213,Up,0,1


In [7]:
X = m3[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5']].values
y = m3['Direction_Up'].values.reshape(-1,1)

lr = LogisticRegression()
lr.fit(X, y)

print(lr.coef_)
print(lr.intercept_)

[[-0.07114368 -0.04402138  0.00921062  0.00719358  0.0092884 ]]
[0.07416154]


C:\Users\rashraf\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:72: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)


In [8]:
#The predict() function: predicts the market "Up" probability on a particular day on the basis of the predicted model

## Sci-Kit learn is using a threshold of P>0.5 for binary classifications
y_pred = lr.predict(X)
y_pred.mean()
y_pred[:5]

array([1, 0, 0, 1, 1], dtype=uint8)

In [9]:
confusion_matrix(y, y_pred)

array([[114, 488],
       [ 98, 550]], dtype=int64)

In [12]:
lr.predict_proba(X)

array([[0.482436  , 0.517564  ],
       [0.5103188 , 0.4896812 ],
       [0.5159246 , 0.4840754 ],
       ...,
       [0.46364239, 0.53635761],
       [0.47184838, 0.52815162],
       [0.4787447 , 0.5212553 ]])

In [21]:
# TO check the actual probability that a data point belongs to a given class, we can use the function predict_proba() 
#The first column corresponds to the probability that the sample belongs to the first class and the second column corresponds to the probability that the sample belongs to the second class.

y_pred=lr.predict_proba(X)[:,1]  #take the second column 

# y_pred is numpy.ndarray. Convert it to pandas.core.series.Series to use the apply function to assign type 1/0 based on probabilities
y_pred_series = pd.Series(y_pred)
y_pred_series.head()
y_pred_series.describe()

type(y_pred_series)

y_pred2 = y_pred_series.apply(lambda r: 1 if r > 0.5 else 0)
y_pred2.head()


y_pred2.describe()
type(y_pred2)
y_pred3 = y_pred2.values.reshape(-1,1)  # change to array 

In [22]:
# Use  confusion matrix to measure the accuracy of our model.

confusion_matrix(y, y_pred3)      


array([[116, 486],
       [ 98, 550]], dtype=int64)

In [23]:
accuracy = np.mean(y_pred3 == y)
print ('accuracy = {0}%'.format(accuracy*100)  )


accuracy = 53.28000000000001%


# Use Lag1 and  Lag2 only as predictors to see if accuracy improves

In [11]:
X = m3[['Lag1', 'Lag2']].values
y = m3['Direction_Up'].values.reshape(-1,1)

lr = LogisticRegression()
lr.fit(X, y)

print(lr.coef_)
print(lr.intercept_)
#The predict() function: predicts the market "Up" probability on a particular day on the basis of the predicted model

## Sci-Kit learn is using a threshold of P>0.5 for binary classifications
y_pred = lr.predict(X)
y_pred.mean()


[[-0.07132639 -0.04437839]]
[0.07401017]


C:\Users\rashraf\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
C:\Users\rashraf\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\utils\validation.py:724: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.8272

In [12]:
# TO check the actual probability that a data point belongs to a given class, we can use the function predict_proba() 
#The first column corresponds to the probability that the sample belongs to the first class and the second column corresponds to the probability that the sample belongs to the second class.

y_pred=lr.predict_proba(X)[:,1]  #take the second column 

# y_pred is numpy.ndarray. Convert it to pandas.core.series.Series to use the apply function to assign type 1/0 based on probabilities
y_pred_series = pd.Series(y_pred)
y_pred_series.head()
y_pred_series.describe()

type(y_pred_series)

y_pred2 = y_pred_series.apply(lambda r: 1 if r > 0.5 else 0)
y_pred2.head()


y_pred2.describe()
type(y_pred2)
y_pred3 = y_pred2.values.reshape(-1,1)  # change to array 
# Use  confusion matrix to measure the accuracy of our model.

confusion_matrix(y, y_pred3)   

array([[114, 488],
       [102, 546]], dtype=int64)

In [13]:
accuracy = np.mean(y_pred3 == y)
print ('accuracy = {0}%'.format(accuracy*100)  )


accuracy = 52.800000000000004%


# Comparison of Model Accuracy 
Model accuracy did not improve when only Lag1 and Lag2 returns are used to predict market movememt.

The model is no better than random guess!
It is difficult to predict future performance using past market performance